# kld_sweep.py — Linux / Colab test
Tests `kld_sweep.py` on **Qwen3.5-0.8B** using two small quants from unsloth.
- BF16: 1.52 GiB
- Quants: UD-Q4_K_XL (559 MB) + UD-IQ2_XXS (338 MB)
- Total disk: ~2.5 GiB

**Runtime:** GPU (T4 recommended). CPU-only will work but logits generation will be slow.

**Steps:** Run all cells in order. The sweep resumes automatically if interrupted.

In [1]:
# @title 1 — Install dependencies
!pip install pandas matplotlib adjustText scipy -q

In [3]:
# @title 2 — Build llama.cpp from source (CUDA, T4 optimized)
# No prebuilt Linux CUDA binary available — build takes ~3-5 minutes
import os

!apt-get install -y cmake libcurl4-openssl-dev -q
!git clone https://github.com/ggml-org/llama.cpp --depth 1 -q
!cmake llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DCMAKE_BUILD_TYPE=Release
!cmake --build llama.cpp/build --target llama-perplexity -j$(nproc)

PPL_EXE = 'llama.cpp/build/bin/llama-perplexity'
assert os.path.exists(PPL_EXE), f'Build failed — {PPL_EXE} not found'
print(f'\nllama-perplexity: {PPL_EXE}')


Reading package lists...
Building dependency tree...
Reading state information...
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
libcurl4-openssl-dev is already the newest version (7.81.0-1ubuntu1.21).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
fatal: destination path 'llama.cpp' already exists and is not an empty directory.
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found ass

In [4]:
# @title 3 — Download BF16 model and quants from HuggingFace
!pip install huggingface_hub -q
from huggingface_hub import hf_hub_download
import os

os.makedirs('bf16',   exist_ok=True)
os.makedirs('quants', exist_ok=True)

REPO = 'unsloth/Qwen3.5-0.8B-GGUF'

bf16_path = hf_hub_download(REPO, 'Qwen3.5-0.8B-BF16.gguf',     local_dir='bf16')
q1_path   = hf_hub_download(REPO, 'Qwen3.5-0.8B-UD-Q4_K_XL.gguf', local_dir='quants')
q2_path   = hf_hub_download(REPO, 'Qwen3.5-0.8B-UD-IQ2_XXS.gguf', local_dir='quants')

print(f'BF16  : {bf16_path}')
print(f'Quant1: {q1_path}')
print(f'Quant2: {q2_path}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Qwen3.5-0.8B-BF16.gguf:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Qwen3.5-0.8B-UD-Q4_K_XL.gguf:   0%|          | 0.00/559M [00:00<?, ?B/s]

Qwen3.5-0.8B-UD-IQ2_XXS.gguf:   0%|          | 0.00/338M [00:00<?, ?B/s]

BF16  : bf16/Qwen3.5-0.8B-BF16.gguf
Quant1: quants/Qwen3.5-0.8B-UD-Q4_K_XL.gguf
Quant2: quants/Qwen3.5-0.8B-UD-IQ2_XXS.gguf


In [5]:
# @title 4 — Download kld_sweep.py and dataset
import os

# Fetch script from GitHub
!wget -q https://raw.githubusercontent.com/cmhamiche/kld-sweep/main/kld_sweep.py
assert os.path.exists('kld_sweep.py'), 'Failed to download kld_sweep.py'
print('kld_sweep.py downloaded')

# Dataset — wikitext2 test set via llama.cpp helper
!wget -q https://raw.githubusercontent.com/ggml-org/llama.cpp/master/scripts/get-wikitext-2.sh
!bash get-wikitext-2.sh
DATASET = 'wikitext-2-raw/wiki.test.raw'
assert os.path.exists(DATASET), f'Dataset not found: {DATASET}'
print(f'Dataset: {DATASET}')
SCRIPT = 'kld_sweep.py'


kld_sweep.py downloaded
--2026-03-04 23:23:33--  https://huggingface.co/datasets/ggml-org/ci/resolve/main/wikitext-2-raw-v1.zip
Resolving huggingface.co (huggingface.co)... 13.35.202.121, 13.35.202.40, 13.35.202.97, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.121|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/65d261003da87ce21e5997f8/b28df100478bded493177f9226b11cd70414e3b60ae217270a1cbdf67fbca0b0?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27wikitext-2-raw-v1.zip%3B+filename%3D%22wikitext-2-raw-v1.zip%22%3B&response-content-type=application%2Fzip&Expires=1772670213&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiRXBvY2hUaW1lIjoxNzcyNjcwMjEzfX0sIlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjVkMjYxMDAzZGE4N2NlMjFlNTk5N2Y4L2IyOGRmMTAwNDc4YmRlZDQ5MzE3N2Y5MjI2YjExY2Q3MDQxNGUzYjYwYWUyMTcyNzBhMWNiZGY2N2ZiY2EwYjBcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0a

In [13]:
# @title 4b — Clean up partial logits if needed
import os

for f in ['output/Qwen3.5-0.8B-logits.bin', 'output/Qwen3.5-0.8B-logits.bin.meta']:
    if os.path.exists(f):
        os.remove(f)
        print(f'Deleted: {f}')

Deleted: output/Qwen3.5-0.8B-logits.bin
Deleted: output/Qwen3.5-0.8B-logits.bin.meta


In [ ]:
# @title 5 — Run sweep
import subprocess, sys, os

cmd = [
    sys.executable, SCRIPT,
    '--exe',        PPL_EXE,
    '--bf16',       bf16_path,
    '--quants',     'quants',
    '--dataset',    DATASET,
    '--output',     'output',
    '--model-name', 'Qwen3.5-0.8B',
    '--args=-t 4 -c 512 -ngl 99',
]

print('Command:', ' '.join(cmd), '\n')

# Stream output live
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, encoding='utf-8', errors='replace')
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

print(f'\nExit code: {proc.returncode}')
if proc.returncode != 0:
    raise RuntimeError(f'kld_sweep.py failed with exit code {proc.returncode}')

print('\nOutput files:')
for f in sorted(os.listdir('output')):
    print(f'  {f}')

Command: /usr/bin/python3 kld_sweep.py --exe llama.cpp/build/bin/llama-perplexity --bf16 bf16/Qwen3.5-0.8B-BF16.gguf --quants quants --dataset wikitext-2-raw/wiki.test.raw --output output --model-name Qwen3.5-0.8B --args=-t 4 -c 512 -ngl 99 


Found 2 quant file(s) in quants

[logits] Generating from Qwen3.5-0.8B-BF16.gguf (1.41 GiB) — ETA will appear below...

>>> Generating logits -> Qwen3.5-0.8B-logits.bin
ggml_cuda_init: found 1 CUDA devices:
  Device 0: Tesla T4, compute capability 7.5, VMM: yes
build: 1 (24d2ee0) with GNU 11.4.0 for Linux x86_64
common_init_result: fitting params to device memory, for bugs during this step try to reproduce them with -fit off, or provide --verbose logs if the bug only occurs with -fit on
llama_params_fit_impl: projected to use 2026 MiB of device memory vs. 14729 MiB of free device memory
llama_params_fit_impl: will leave 12703 >= 1024 MiB of free device memory, no changes needed
llama_params_fit: successfully fit params to free device memory
llama

In [ ]:
# @title 6 — Show results
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv('output/Qwen3.5-0.8B_results.csv')
display(df.sort_values('KLD_Score'))

display(Image('output/kld_plot_Qwen3.5-0.8B.png'))
display(Image('output/ppl_plot_Qwen3.5-0.8B.png'))